# 28 — Classic LeetCode: Top 5 Problems

**Target:** generic FAANG-style screens (Shield AI, Vantor, Helsing, Striveworks, etc).
Notebook 27 covers ServiceTitan's HackerRank-flavored screen; this one is the
broad portable set.

The 5 here cover the **core patterns** that show up in 80% of company screens:

| # | Problem | Pattern |
|---|---------|---------|
| 1 | Two Sum | Hash map / one-pass complement lookup |
| 2 | Valid Parentheses | Stack |
| 3 | Merge Intervals | Sort + linear scan |
| 4 | LRU Cache (OrderedDict version) | Cache eviction policy — idiomatic Python |
| 5 | Course Schedule | Topological sort / cycle detection in DAG |

> Note: Notebook 27 also implements LRU but from scratch with linked-list. Here
> we show the **idiomatic OrderedDict version** for the same problem — a real
> interviewer often asks both. Knowing both signals depth.

Each problem: statement → naive/optimal approaches → solution → 10 test cases.


---
## Problem 1 — Two Sum

> Given `nums` and `target`, return indices `i, j` such that `nums[i] + nums[j] == target`.
> Each input has exactly one solution. You may not use the same element twice.

### Approaches

| Approach | Time | Space | Notes |
|----------|------|-------|-------|
| Brute force double loop | O(n²) | O(1) | The "warm-up" answer |
| Sort + two-pointer | O(n log n) | O(n) (need original indices) | Loses index info — has to re-find |
| **Hash map, one-pass** (chosen) | **O(n)** | O(n) | Look up complement as you scan |

### Insight

As you scan, for each `x` check if `target - x` was already seen. If yes, return both
indices. If not, store `x → index`. This is the canonical interview answer — interviewers
specifically watch for the one-pass version vs. the build-then-scan two-pass version.


In [ ]:
from typing import List


def two_sum(nums: List[int], target: int) -> List[int]:
    """Return indices i, j with nums[i] + nums[j] == target. One-pass hash map."""
    seen: dict[int, int] = {}   # value -> index
    for i, x in enumerate(nums):
        complement = target - x
        if complement in seen:
            return [seen[complement], i]
        seen[x] = i
    raise ValueError("no two-sum solution exists")


In [ ]:
# ── Test cases for Problem 1 ──────────────────────────────────────────────

# 1. canonical example
assert sorted(two_sum([2, 7, 11, 15], 9)) == [0, 1]

# 2. solution not at start
assert sorted(two_sum([3, 2, 4], 6)) == [1, 2]

# 3. duplicates — same value, different indices
assert sorted(two_sum([3, 3], 6)) == [0, 1]

# 4. negative numbers
assert sorted(two_sum([-1, -2, -3, -4, -5], -8)) == [2, 4]

# 5. mixed signs
assert sorted(two_sum([-3, 4, 3, 90], 0)) == [0, 2]

# 6. zero target with zero element
assert sorted(two_sum([0, 4, 3, 0], 0)) == [0, 3]

# 7. large numbers - algorithm returns FIRST matching pair encountered
big = 10**9
# scanning: big->complement=0 miss, (big-5)->5 miss, 5->big-5 HIT at idx 1
assert sorted(two_sum([big, big - 5, 5, 0], big)) == [1, 2]

# 8. solution at last two indices (worst case for one-pass exit)
# [1,2,3,4,5,5] target 10 — only pair summing to 10 is the two 5s
assert sorted(two_sum([1, 2, 3, 4, 5, 5], 10)) == [4, 5]

# 9. no solution raises (don't silently return garbage)
try:
    two_sum([1, 2, 3], 100)
    assert False, "should have raised"
except ValueError:
    pass

# 10. cannot reuse the same element twice
# input has 3 once; target 6 would tempt 3+3, but only one 3 exists
try:
    two_sum([3, 1, 4], 6)
    assert False, "should not reuse element"
except ValueError:
    pass

print("✓ Problem 1 (Two Sum) — all 10 test cases pass")


---
## Problem 2 — Valid Parentheses

> Given a string containing `()[]{}`, determine if it's valid. Brackets must
> close in the correct order: every opener has a matching closer of the same
> kind, and they must nest properly.

### Approach

The structure of nested matching pairs is exactly what a **stack** models. Push
openers, on each closer check the top of stack matches. Three failure modes:
- Closer with empty stack (unmatched closer)
- Closer doesn't match top of stack (wrong kind)
- Stack non-empty at end (unclosed opener)

Time: O(n). Space: O(n) for the stack.


In [ ]:
def is_valid_parens(s: str) -> bool:
    """True iff every bracket in s is properly matched and nested."""
    pairs = {")": "(", "]": "[", "}": "{"}
    openers = set(pairs.values())
    stack: list[str] = []
    for ch in s:
        if ch in openers:
            stack.append(ch)
        elif ch in pairs:
            if not stack or stack[-1] != pairs[ch]:
                return False
            stack.pop()
        # non-bracket chars: ignore (some interview variants permit, some don't)
    return not stack


In [ ]:
# ── Test cases for Problem 2 ──────────────────────────────────────────────

# 1. basic balanced
assert is_valid_parens("()")
assert is_valid_parens("()[]{}")

# 2. nested
assert is_valid_parens("([{}])")

# 3. unbalanced — wrong order
assert not is_valid_parens("(]")

# 4. unbalanced — only opener
assert not is_valid_parens("(")

# 5. unbalanced — only closer
assert not is_valid_parens(")")

# 6. empty string is valid (trivially balanced)
assert is_valid_parens("")

# 7. mismatched nesting
assert not is_valid_parens("([)]")
assert is_valid_parens("([])")

# 8. deep nesting (1000 levels)
deep = "(" * 1000 + ")" * 1000
assert is_valid_parens(deep)

# 9. single bracket type, repeated
assert is_valid_parens("()()()()()")
assert not is_valid_parens("(()")

# 10. adversarial — long string, fails on last char
s = "(" * 500 + ")" * 499 + "]"
assert not is_valid_parens(s)

print("✓ Problem 2 (Valid Parentheses) — all 10 test cases pass")


---
## Problem 3 — Merge Intervals

> Given a list of intervals `[[start, end], ...]`, merge all overlapping
> intervals and return the result.

### Approaches

| Approach | Time | Space |
|----------|------|-------|
| For each pair check overlap, repeat until stable | O(n²) worst case | O(n) |
| **Sort by start, then linear scan** (chosen) | O(n log n) | O(n) |

### Insight

After sorting by start, two adjacent intervals `[a, b]` and `[c, d]` overlap iff
`c <= b`. If they overlap, merge into `[a, max(b, d)]`. If not, the first is finalized.

This is the textbook "interval scheduling" pattern — appears in calendar merging,
process scheduling, genome assembly. Worth memorizing.


In [ ]:
from typing import List


def merge_intervals(intervals: List[List[int]]) -> List[List[int]]:
    """Merge overlapping intervals. Returns sorted list of non-overlapping merged intervals."""
    if not intervals:
        return []
    # validate
    for iv in intervals:
        if len(iv) != 2 or iv[0] > iv[1]:
            raise ValueError(f"invalid interval {iv}")

    sorted_iv = sorted(intervals, key=lambda x: x[0])
    merged: list[list[int]] = [list(sorted_iv[0])]

    for start, end in sorted_iv[1:]:
        if start <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])
    return merged


In [ ]:
# ── Test cases for Problem 3 ──────────────────────────────────────────────

# 1. canonical example
assert merge_intervals([[1, 3], [2, 6], [8, 10], [15, 18]]) == [[1, 6], [8, 10], [15, 18]]

# 2. touching intervals merge (boundary inclusive)
assert merge_intervals([[1, 4], [4, 5]]) == [[1, 5]]

# 3. nested interval
assert merge_intervals([[1, 10], [2, 5], [3, 4]]) == [[1, 10]]

# 4. already-sorted, no overlaps
assert merge_intervals([[1, 2], [3, 4], [5, 6]]) == [[1, 2], [3, 4], [5, 6]]

# 5. empty input
assert merge_intervals([]) == []

# 6. single interval
assert merge_intervals([[5, 9]]) == [[5, 9]]

# 7. reverse-sorted input
assert merge_intervals([[15, 18], [8, 10], [2, 6], [1, 3]]) == [[1, 6], [8, 10], [15, 18]]

# 8. all merge into one
assert merge_intervals([[1, 5], [2, 3], [4, 8], [6, 10]]) == [[1, 10]]

# 9. invalid interval raises
try:
    merge_intervals([[5, 3]]); assert False
except ValueError:
    pass

# 10. adversarial — single-point intervals
assert merge_intervals([[1, 1], [2, 2], [1, 2]]) == [[1, 2]]

print("✓ Problem 3 (Merge Intervals) — all 10 test cases pass")


---
## Problem 4 — LRU Cache (OrderedDict idiomatic version)

> Same problem as notebook 27 problem 2, but **using `OrderedDict`** instead of
> hand-rolled doubly-linked list. The OrderedDict version is what you reach for
> in real Python code; the linked-list version is what HackerRank wants for the
> screening problem.

### Why include both?

Real interviewers often ask: "now do it idiomatically in Python." If you only know
the from-scratch version, you'll over-engineer it. If you only know `OrderedDict`,
you can't answer "what data structure underlies this?"

### Approach

`OrderedDict.move_to_end(key)` is O(1) and gives us the LRU semantics directly.
On `get`, move the key to end (most-recent). On `put`, if full, `popitem(last=False)`
evicts the front (least-recent).


In [ ]:
from collections import OrderedDict
from typing import Any


class LRUCacheOD:
    """LRU Cache using OrderedDict — idiomatic Python version."""

    def __init__(self, capacity: int) -> None:
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.cap = capacity
        self._cache: "OrderedDict[Any, Any]" = OrderedDict()

    def get(self, key: Any) -> Any:
        if key not in self._cache:
            return -1
        self._cache.move_to_end(key)
        return self._cache[key]

    def put(self, key: Any, value: Any) -> None:
        if key in self._cache:
            self._cache.move_to_end(key)
        elif len(self._cache) >= self.cap:
            self._cache.popitem(last=False)  # evict LRU
        self._cache[key] = value

    def __len__(self) -> int:
        return len(self._cache)

    def __repr__(self) -> str:
        return f"LRUCacheOD({list(self._cache.keys())})"


In [ ]:
# ── Test cases for Problem 4 ──────────────────────────────────────────────
# (Mirror notebook 27's LRU tests to verify behavioral equivalence.)

# 1. basic put + get
c = LRUCacheOD(2)
c.put(1, "a"); c.put(2, "b")
assert c.get(1) == "a"

# 2. miss returns -1
assert c.get(99) == -1

# 3. capacity-1 eviction
c = LRUCacheOD(1)
c.put("x", 1); c.put("y", 2)
assert c.get("x") == -1 and c.get("y") == 2

# 4. get refreshes recency
c = LRUCacheOD(2)
c.put(1, "a"); c.put(2, "b")
_ = c.get(1)
c.put(3, "c")
assert c.get(1) == "a" and c.get(2) == -1 and c.get(3) == "c"

# 5. update doesn't evict
c = LRUCacheOD(2)
c.put(1, "a"); c.put(2, "b"); c.put(1, "A")
assert c.get(1) == "A" and c.get(2) == "b"

# 6. update refreshes recency
c = LRUCacheOD(2)
c.put(1, "a"); c.put(2, "b"); c.put(1, "A")
c.put(3, "c")
assert c.get(2) == -1 and c.get(1) == "A"

# 7. invalid capacity raises
try:
    LRUCacheOD(0); assert False
except ValueError:
    pass

# 8. heavy churn — capacity invariant
c = LRUCacheOD(3)
for i in range(1000):
    c.put(i, i * 2)
    assert len(c) <= 3

# 9. cycle pattern — verify against notebook 27's expected result
c = LRUCacheOD(3)
for v in [1, 2, 3, 4, 1, 2, 3, 4, 1]:
    c.put(v, v)
assert c.get(3) == 3 and c.get(4) == 4 and c.get(1) == 1
assert c.get(2) == -1

# 10. behavioral equivalence: random ops match dict-of-truth on small capacity
import random
random.seed(42)
truth = {}
recency = []  # insertion order tracker for capacity 5
cap = 5
c = LRUCacheOD(cap)
for _ in range(200):
    if random.random() < 0.5 and recency:
        k = random.choice(recency)
        got = c.get(k)
        assert got == truth.get(k, -1), f"get({k})={got} but truth={truth.get(k)}"
        # refresh recency
        recency.remove(k); recency.append(k)
    else:
        k = random.randint(0, 20)
        v = random.randint(0, 1000)
        if k in truth:
            recency.remove(k)
        elif len(truth) >= cap:
            lru_key = recency.pop(0)
            del truth[lru_key]
        truth[k] = v
        recency.append(k)
        c.put(k, v)

print("✓ Problem 4 (LRU OrderedDict) — all 10 test cases pass")


---
## Problem 5 — Course Schedule (Topological Sort / Cycle Detection)

> `numCourses` courses labeled `0..n-1`. `prerequisites[i] = [a, b]` means you
> must take `b` before `a`. Return `True` iff you can finish all courses.

Equivalent to: **does this directed graph have a cycle?** If yes, impossible.

### Approaches

| Approach | Time | Notes |
|----------|------|-------|
| DFS with 3-color visited (chosen) | O(V + E) | Detect back edges = cycle |
| Kahn's algorithm (BFS, in-degree) | O(V + E) | Returns valid order too |

### Insight (DFS coloring)

Three states per node:
- **WHITE** (0): unvisited
- **GRAY** (1): on current DFS stack
- **BLACK** (2): fully processed

If you ever encounter a GRAY neighbor, you've found a back edge → cycle. This is
the cleanest way to detect cycles in a directed graph.

We'll implement DFS coloring; it generalizes well to "return the order" follow-ups.


In [ ]:
from typing import List
from collections import defaultdict


def can_finish(num_courses: int, prerequisites: List[List[int]]) -> bool:
    """True iff the prerequisite DAG has no cycle (i.e., all courses are completable)."""
    if num_courses <= 0:
        return True
    graph: dict = defaultdict(list)
    for a, b in prerequisites:
        if not (0 <= a < num_courses and 0 <= b < num_courses):
            raise ValueError(f"course id out of range in {[a, b]}")
        graph[b].append(a)   # b -> a (must take b before a)

    WHITE, GRAY, BLACK = 0, 1, 2
    color = [WHITE] * num_courses

    def dfs(u: int) -> bool:
        if color[u] == GRAY:
            return False   # back edge → cycle
        if color[u] == BLACK:
            return True
        color[u] = GRAY
        for v in graph[u]:
            if not dfs(v):
                return False
        color[u] = BLACK
        return True

    return all(dfs(u) for u in range(num_courses))


In [ ]:
# ── Test cases for Problem 5 ──────────────────────────────────────────────
import sys
sys.setrecursionlimit(50_000)

# 1. simple linear chain
assert can_finish(2, [[1, 0]])

# 2. simple cycle
assert not can_finish(2, [[1, 0], [0, 1]])

# 3. no prereqs at all
assert can_finish(5, [])

# 4. self-loop is a cycle
assert not can_finish(3, [[1, 1]])

# 5. disconnected components — one good, one cyclic
assert not can_finish(4, [[1, 0], [3, 2], [2, 3]])

# 6. disconnected — all good
assert can_finish(4, [[1, 0], [3, 2]])

# 7. diamond DAG — multiple paths but no cycle
#    0 -> 1, 0 -> 2, 1 -> 3, 2 -> 3
assert can_finish(4, [[1, 0], [2, 0], [3, 1], [3, 2]])

# 8. longer cycle (3 nodes)
assert not can_finish(3, [[1, 0], [2, 1], [0, 2]])

# 9. deep chain — performance / recursion depth
chain = [[i + 1, i] for i in range(1000)]
assert can_finish(1001, chain)

# 10. adversarial — many edges, single back-edge buried deep
edges = [[i + 1, i] for i in range(99)]   # chain 0→1→...→99, 100 nodes
edges.append([0, 99])   # closes a cycle 0→1→...→99→0
assert not can_finish(100, edges)

# Bonus: invalid course id raises
try:
    can_finish(2, [[5, 0]]); assert False
except ValueError:
    pass

print("✓ Problem 5 (Course Schedule) — all 10 test cases pass")


---
## Final Sanity Check — all 5 problems together


In [ ]:
print("=" * 60)
print("Notebook 28 — Classic LeetCode Top 5")
print("=" * 60)
print(f"  Problem 1 (Two Sum):              PASS")
print(f"  Problem 2 (Valid Parentheses):    PASS")
print(f"  Problem 3 (Merge Intervals):      PASS")
print(f"  Problem 4 (LRU OrderedDict):      PASS")
print(f"  Problem 5 (Course Schedule):      PASS")
print("=" * 60)
print("All 50 test cases passing.")
